In [ ]:
!pip install autoawq datasets transformers
!pip install autoawq-kernels
!pip install bitsandbytes
!pip install tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 3.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for autoawq: filename=autoawq-0.2.9-py3-none-any.whl size=115106 sha256=552776ee59e832e3f98ecaa38d326ad82c4643c05f26b828fcd5805b391fc02a
  Stored in directory: /root/.cache/pip/wheels/45/1a/7b/7314b3a958454e8ce349f600829a3f0a6a05aeebf987be1e16
Successfully built autoawq


In [ ]:
import torch
from awq import AutoAWQForCausalLM
from transformers import AutoTokenizer
from datasets import load_dataset

/usr/local/lib/python3.12/dist-packages/awq/__init__.py:21: DeprecationWarning: 
I have left this message as the final dev message to help you transition.

Important Notice:
- AutoAWQ is officially deprecated and will no longer be maintained.
- The last tested configuration used Torch 2.6.0 and Transformers 4.51.3.
- If future versions of Transformers break AutoAWQ compatibility, please report the issue to the Transformers project.

Alternative:
- AutoAWQ has been adopted by the vLLM Project: https://github.com/vllm-project/llm-compressor

For further inquiries, feel free to reach out:
- X: https://x.com/casper_hansen_
- LinkedIn: https://www.linkedin.com/in/casper-hansen-804005170/

  warnings.warn(_FINAL_DEV_MESSAGE, category=DeprecationWarning, stacklevel=1)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes

In [ ]:
from transformers import BitsAndBytesConfig
from tqdm import tqdm

In [ ]:
from google.colab import drive
import os

# 1. Mount Google Drive
# This will prompt you to click a link and authorize access
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
drive_model_folder = "Mistral-7B-v0.3-AWQ-4bit-Wikitext"
model_path = f"/content/drive/My Drive/{drive_model_folder}"

# 3. Verify the path exists and list files
if os.path.exists(model_path):
    print(f"✅ Model found at: {model_path}")
    print("Files in directory:", os.listdir(model_path))
else:
    print(f"❌ Error: Could not find the folder at {model_path}")
    print("Please check your Google Drive folder name and structure.")

✅ Model found at: /content/drive/My Drive/Mistral-7B-v0.3-AWQ-4bit-Wikitext
Files in directory: ['generation_config.json', 'config.json', 'model.safetensors', 'tokenizer_config.json', 'tokenizer.json']


In [ ]:
# New version
def calculate_perplexity(model_path):
    device = "cuda"

    # 1. Load Model & Tokenizer
    print(f"📦 Loading Model...")
    model = AutoAWQForCausalLM.from_quantized(model_path, fuse_layers=False).to(device)
    tokenizer = AutoTokenizer.from_pretrained(model_path)

    # 2. Dataset Setup
    test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    encodings = tokenizer("\n\n".join(test_data["text"]), return_tensors="pt")

    # 3. Parameters
    max_length = 2048
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nll_sum = 0.0
    n_tokens = 0

    print(f"📊 Running Sliding Window Evaluation...")
    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - begin_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()

        # Mask the context tokens so we only calculate loss on the new ones
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)

            # Use float32 to prevent overflow during summation
            loss = outputs.loss.to(torch.float32)

            # Check for invalid values that cause 'inf'
            if torch.isnan(loss) or torch.isinf(loss):
                continue

            nll_sum += loss * trg_len
            n_tokens += trg_len

        if end_loc == seq_len:
            break

    # 4. Final Math
    # Calculate average negative log-likelihood
    avg_nll = nll_sum / n_tokens

    # Safety clamp: if nll is too high, exp() will still return inf
    if avg_nll > 20:
        print("⚠️ Warning: Average Loss is extremely high. Model may be corrupted.")

    ppl = torch.exp(avg_nll)
    return ppl.item()

# Execute
#model_drive_path = "/content/drive/My Drive/Mistral-7B-v0.3-AWQ-4bit-Wikitext"


In [ ]:
ppl_result = calculate_perplexity(model_path)
print(f"\n✨ Final Perplexity: {ppl_result:.4f}")

📦 Loading Model...


Replacing layers...: 100%|██████████| 32/32 [00:10<00:00,  3.12it/s]


📊 Running Sliding Window Evaluation...


 99%|█████████▉| 650/654 [37:53<00:13,  3.50s/it]


✨ Final Perplexity: 5.4480


In [ ]:

def calculate_perplexity_baseline_logic(model, tokenizer):
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1. Load WikiText-2 Test Set (Exactly the same as Phase 2)
    test_data = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
    encodings = tokenizer("\n\n".join(test_data["text"]), return_tensors="pt")

    # 2. Parameters (Must match your quantized test for a fair comparison)
    max_length = 1024
    stride = 512
    seq_len = encodings.input_ids.size(1)

    nll_sum = 0.0
    n_tokens = 0

    print(f"📊 Running Baseline FP16 Evaluation...")
    model.eval()

    for begin_loc in tqdm(range(0, seq_len, stride)):
        end_loc = min(begin_loc + max_length, seq_len)
        trg_len = end_loc - begin_loc

        input_ids = encodings.input_ids[:, begin_loc:end_loc].to(device)
        target_ids = input_ids.clone()

        # Mask context tokens (match the logic used for the quantized model)
        target_ids[:, :-trg_len] = -100

        with torch.no_grad():
            outputs = model(input_ids, labels=target_ids)

            # Sum up Negative Log-Likelihood
            loss = outputs.loss.to(torch.float32)
            nll_sum += loss * trg_len
            n_tokens += trg_len

        if end_loc == seq_len:
            break

    avg_nll = nll_sum / n_tokens
    ppl = torch.exp(avg_nll)
    return ppl.item()

# --- HOW TO RUN ---
# from transformers import AutoModelForCausalLM, AutoTokenizer

# baseline_model_id = "mistralai/Mistral-7B-v0.3"
# tokenizer = AutoTokenizer.from_pretrained(baseline_model_id)
# model = AutoModelForCausalLM.from_pretrained(
#     baseline_model_id,
#     torch_dtype=torch.float16,
#     device_map="auto"
# )

# ppl_baseline = calculate_perplexity_baseline_logic(model, tokenizer)
# print(f"📉 Baseline PPL: {ppl_baseline:.4f}")

In [ ]:
# Baseline Script (Mistral-7B-v0.3 FP16)
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# 1. Configuration
baseline_model_id = "mistralai/Mistral-7B-v0.3"
tokenizer = AutoTokenizer.from_pretrained(baseline_model_id)
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True # Extra safety for OOM
)

# 2. Load the Baseline in FP8
# device_map="auto" will handle the 15GB load
print("📦 Loading Baseline FP8 Model...")
baseline_model = AutoModelForCausalLM.from_pretrained(
    baseline_model_id,

    quantization_config=quantization_config,
    device_map="auto",
    low_cpu_mem_usage=True
)

# 3. Use your Phase 2 Function
# Simply call the function we wrote earlier, but pass this model
# Note: You'll need to adjust your function slightly to accept an object OR path
# For now, just re-run your Phase 2 script logic with this 'baseline_model'
ppl_baseline = calculate_perplexity_baseline_logic(baseline_model, tokenizer)

print(f"📉 Baseline FP8 PPL: {ppl_baseline:.4f}")
print(f"📊 Quantization Gap: {5.4480 - ppl_baseline:.4f}")
# baseline FP8 PPL: 5.955, Quantization Gap: -0.5075

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/601 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

...

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

📊 Running Baseline FP16 Evaluation...


  0%|          | 0/654 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.bfloat16 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
100%|█████████▉| 652/654 [38:20<00:07,  3.53s/it]

📉 Baseline FP8 PPL: 5.9555
📊 Quantization Gap: -0.5075


In [ ]:
import torch
import time
from transformers import AutoTokenizer
from awq import AutoAWQForCausalLM # Use standard AutoModelForCausalLM for baseline

def measure_tps(model_path, is_quantized=True, prompt="Explain the theory of relativity in detail."):
    device = "cuda"

    # 1. Load Model & Tokenizer
    print(f"📦 Loading {'Quantized' if is_quantized else 'Baseline'} Model...")
    if is_quantized:
        #model = AutoAWQForCausalLM.from_quantized(model_path, fuse_layers=True).to(device)
        model = AutoAWQForCausalLM.from_quantized(
           model_path,
           fuse_layers=True,           # This combines multiple layers into one GPU call
           return_as_trainable=False,
           use_cache=True,
           #attn_implementation="flash_attention_2"
        )
    else:
        # For baseline, ensure you use the 8-bit config we discussed to avoid OOM
        from transformers import AutoModelForCausalLM, BitsAndBytesConfig
        model = AutoModelForCausalLM.from_pretrained(
            model_path,
            quantization_config=BitsAndBytesConfig(load_in_8bit=True),
            device_map="auto"
        )
    model.eval()
    tokenizer = AutoTokenizer.from_pretrained(model_path)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 2. Warm-up (Ensures CUDA kernels are initialized)
    _ = model.generate(**inputs, max_new_tokens=5)

    # 3. Benchmark Generation
    max_new_tokens = 200
    print(f"🚀 Generating {max_new_tokens} tokens...")

    torch.cuda.synchronize() # Wait for GPU to be ready
    start_time = time.perf_counter()

    output_tokens = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=True,
        temperature=0.7,
        use_cache=True # Vital for speed!
    )

    torch.cuda.synchronize() # Wait for GPU to finish
    end_time = time.perf_counter()

    # 4. Math
    total_time = end_time - start_time
    # We only care about the NEW tokens generated
    new_tokens_count = len(output_tokens[0]) - len(inputs.input_ids[0])
    tps = new_tokens_count / total_time

    print(f"\n--- Results ---")
    print(f"⏱️ Total Time: {total_time:.2f} seconds")
    print(f"📝 Tokens Generated: {new_tokens_count}")
    print(f"⚡ Throughput: {tps:.2f} tokens/sec")

    return tps

# Run Test
#tps_baseline = measure_tps("mistralai/Mistral-7B-v0.3", is_quantized=False)
tps_quant = measure_tps("/content/drive/My Drive/Mistral-7B-v0.3-AWQ-4bit-Wikitext", is_quantized=True)

📦 Loading Quantized Model...


`torch_dtype` is deprecated! Use `dtype` instead!
Replacing layers...: 100%|██████████| 32/32 [00:08<00:00,  3.60it/s]
/usr/local/lib/python3.12/dist-packages/awq/models/base.py:541: UserWarning: Skipping fusing modules because AWQ extension is not installed./usr/local/lib/python3.12/dist-packages/awq_ext.cpython-312-x86_64-linux-gnu.so: undefined symbol: _ZN3c104cuda29c10_cuda_check_implementationEiPKcS2_ib
  warnings.warn("Skipping fusing modules because AWQ extension is not installed." + msg)
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


🚀 Generating 200 tokens...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)



--- Results ---
⏱️ Total Time: 40.10 seconds
📝 Tokens Generated: 200
⚡ Throughput: 4.99 tokens/sec


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
--- BL Results ---
⏱️ Total Time: 41.71 seconds
📝 Tokens Generated: 200
⚡ Throughput: 4.80 tokens/sec

--- Results ---
⏱️ Total Time: 40.10 seconds
📝 Tokens Generated: 200
⚡ Throughput: 4.99 tokens/sec

